# Exercise A — Regression, Gradient Descent, and a Tiny Neural Network

**Companion to Lecture A** of the lecture notes *Essential Concepts in FEM and Physics-Based ML*.

How to work: fill in every `...` (TODO), run the ✅ check cell after each part.
Parts build on each other — don't skip ahead.

---
*Author: Ramses Sala · Copyright © 2025 Ramses Sala · Part of the course* Essential Concepts in FEM and Physics-Based ML. *Created with assistance of Claude Code for text, image and code refinement and generation.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# ---------- helpers provided for all exercises (read them once!) ----------

# num_grad is a transparent stand-in for automatic differentiation: real frameworks
# compute the same numbers exactly and much faster.
def num_grad(f, th, h=1e-6):
    """Numerical gradient of scalar f at parameter vector th (central differences)."""
    g = np.zeros_like(th)
    for i in range(len(th)):
        tp = th.copy(); tm = th.copy(); tp[i] += h; tm[i] -= h
        g[i] = (f(tp) - f(tm)) / (2*h)
    return g

def adam_minimize(f, th0, steps=2000, lr=0.02, verbose_every=None):
    """Adam: gradient descent with per-parameter step adaptation (industry standard)."""
    th = np.asarray(th0, float).copy()
    m = np.zeros_like(th); v = np.zeros_like(th)
    for k in range(steps):
        g = num_grad(f, th)
        m = 0.9*m + 0.1*g
        v = 0.999*v + 0.001*g*g
        mh = m/(1-0.9**(k+1)); vh = v/(1-0.999**(k+1))
        th -= lr*mh/(np.sqrt(vh)+1e-8)
        if verbose_every and k % verbose_every == 0:
            print(f"step {k:5d}   J = {f(th):.4e}")
    return th


---
## Part 1 — Least-squares regression by normal equations

Data: $(0,1),\,(1,2),\,(2,5)$. Model: $\tilde y(x) = a_1 + a_2 x^2$.

1. Build the matrix $\mathbf A$ with $A_{ij} = \varphi_j(x_i)$ for the basis $\{1, x^2\}$.
2. Solve the normal equations $(\mathbf A^T\mathbf A)\,\mathbf a = \mathbf A^T \mathbf y$ **by hand first**
   (the system is $2\times2$ with integer entries), then confirm in code.
3. Add the fourth point $(3, 9)$, which lies *off* the parabola, and re-solve.
   Compute the residual vector $\mathbf r = \mathbf A\mathbf a - \mathbf y$.

In [ ]:
x3 = np.array([0., 1., 2.]); y3 = np.array([1., 2., 5.])
A3 = ...            # TODO: columns [1, x^2]
a3 = ...            # TODO: solve the normal equations

x4 = np.array([0., 1., 2., 3.]); y4 = np.array([1., 2., 5., 9.])
A4 = ...
a4 = ...
r4 = ...            # TODO: residual vector
print("a3 =", a3, "   a4 =", a4, "   residuals:", np.round(r4, 3))

In [ ]:
# ✅ check — Part 1
assert np.allclose(a3, [1, 1]), "3 points on the curve: the fit should be exact, a = [1, 1]."
assert np.allclose(a4, [8/7, 87/98]), "4th point off the curve: expect a = [8/7, 87/98]."
assert abs(np.sum(r4)) < 1e-10, "Least squares balances the residuals (they sum to ~0 when a constant is in the basis)."
print("OK. Note: 3 points -> zero residual (truth inside the family); 4 points -> compromise.")

---
## Part 2 — Residual versus error (Lecture A, Section A.3)

Setting of the notes (units $EA = L = n_0 = 1$): fixed–fixed bar, linear load $n(x) = x$,
one-term collocation solution $\tilde u = \tfrac14 x(1-x)$ (collocated at $x = \tfrac12$),
exact solution $u_{\rm ex} = \tfrac16 x(1 - x^2)$.

1. Compute symbolically the **residual** $R(x) = \tilde u'' + n(x)$ and the **error** $e(x) = \tilde u - u_{\rm ex}$.
2. Verify the identity $e'' = R$ (with $EA=1$): *the error is the response to the residual-load*.
3. Plot both. In a markdown cell: what are the physical **units** of each (if $EA$, $n_0$ carried units)?

In [ ]:
xs = sp.Symbol('x')
u_t  = sp.Rational(1, 4) * xs * (1 - xs)
u_ex = xs * (1 - xs**2) / 6

R = ...     # TODO: residual of  u'' + x = 0
e = ...     # TODO: error

print("R =", sp.simplify(R), "    e =", sp.factor(sp.simplify(e)))
f_R = sp.lambdify(xs, R); f_e = sp.lambdify(xs, e)
xx = np.linspace(0, 1, 200)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(xx, f_R(xx), 'r'); ax[0].set_title('residual R(x)  [force/length]')
ax[1].plot(xx, f_e(xx), 'b'); ax[1].set_title('error e(x)  [displacement]')
plt.tight_layout(); plt.show()

In [ ]:
# ✅ check — Part 2
assert sp.simplify(R - (xs - sp.Rational(1, 2))) == 0, "Expect R = x - 1/2."
assert sp.simplify(e - xs*(1 - xs)*(1 - 2*xs)/12) == 0, "Expect e = x(1-x)(1-2x)/12."
assert sp.simplify(sp.diff(e, xs, 2) - R) == 0, "The identity e'' = R must hold (EA = 1)."
print("OK. R vanishes only at the collocation point x = 1/2; e vanishes at the supports.")

---
## Part 3 — Nonlinear regression by gradient descent

Fit $\tilde y = e^{bx}$ to the data $(0,1)$, $(1,2)$. The loss reduces to $J(b) = (e^b - 2)^2$
(why does the first point drop out?). Exact minimum: $b = \ln 2$.

1. Derive $J'(b)$ on paper.
2. Implement plain gradient descent, $\eta = 0.1$, from $b^0 = 0$; print a small table of $(k, b, J)$.
3. **Experiment**: raise $\eta$. Near which value does the iteration start to diverge?
   (Hint: the local curvature at the minimum is $J''(\ln 2) = 8$; GD is stable for $\eta < 2/J''$.)

In [ ]:
b = 0.0
eta = 0.1
for k in range(60):
    if k < 6: print(f"k={k}  b={b:.4f}  J={(np.exp(b)-2)**2:.4f}")
    grad = ...     # TODO: J'(b)
    b = ...        # TODO: descent step
print("final b =", b, "   ln 2 =", np.log(2))

In [ ]:
# ✅ check — Part 3
assert abs(b - np.log(2)) < 1e-3, "After 60 steps with eta=0.1 you should be within 1e-3 of ln 2."
print("OK. Compare with Part 1: same kind of loss, but no normal equations exist here — only the walk.")

---
## Part 4 — A tiny neural network as an adjustable ansatz

Below, a one-hidden-layer tanh network (given, with exact derivatives — read it!).
We fit it to 12 noisy samples of the Lecture-2 solution $u(x) = x - x^2/2$ — *pure regression,
no physics yet*.

1. Write the mean-squared data loss.
2. Train with `adam_minimize` (H = 5 units, ~1500 steps).
3. Plot the fit against the truth, and compute both the final **data residual** (RMS) and the
   **true error** (RMS against $x - x^2/2$ on a dense grid). Which is smaller? Why?

In [ ]:
# ---------- a tiny tanh network, with exact derivatives (given) ----------
# g(x) = sum_j c_j * tanh(w_j x + b_j) + c0          parameters th = [c (H), w (H), b (H), c0]

def net(x, th, H):
    """Returns g(x), g'(x), g''(x) for the one-hidden-layer tanh network."""
    c = th[:H]; w = th[H:2*H]; b = th[2*H:3*H]; c0 = th[3*H]
    t  = np.tanh(np.outer(np.atleast_1d(x), w) + b)
    s2 = 1 - t**2                       # sech^2
    g   = t @ c + c0
    gp  = (s2 * w) @ c
    gpp = (-2 * t * s2 * w**2) @ c
    return g, gp, gpp

def u_hard(x, th, H):
    """u(x) = x * g(x): essential BC u(0)=0 built in (HARD). Returns u, u', u''."""
    x = np.atleast_1d(x).astype(float)
    g, gp, gpp = net(x, th, H)
    return x*g, g + x*gp, 2*gp + x*gpp

def init_net(H, seed=0):
    rng = np.random.default_rng(seed)
    return np.concatenate([rng.normal(0, 0.1, H), rng.normal(0, 1, H),
                           rng.normal(0, 1, H), [0.0]])


In [ ]:
# data (given): noisy samples of u(x) = x - x^2/2
rng = np.random.default_rng(1)
x_dat = np.linspace(0, 1, 12)
y_dat = (x_dat - x_dat**2/2) + 0.01*rng.standard_normal(12)
H = 5

In [ ]:
def data_loss(th):
    g, _, _ = net(x_dat, th, H)
    return ...                      # TODO: mean squared data residual

th = adam_minimize(data_loss, init_net(H, seed=0), steps=1500, lr=0.05, verbose_every=500)

xx = np.linspace(0, 1, 300)
g_fit, _, _ = net(xx, th, H)
res_rms = ...                       # TODO: RMS of (fit - data) at the data points
err_rms = ...                       # TODO: RMS of (fit - truth) on the dense grid
print(f"data residual RMS = {res_rms:.4f}   true error RMS = {err_rms:.4f}")
plt.plot(xx, xx - xx**2/2, 'b', label='truth')
plt.plot(xx, g_fit, 'r--', label='network fit')
plt.plot(x_dat, y_dat, 'k.', label='noisy data'); plt.legend(); plt.show()

In [ ]:
# ✅ check — Part 4
assert res_rms < 0.03 and err_rms < 0.03, "Both RMS values should be small (approx. the noise level)."
print("OK. 'Training a network' was nothing but Part 3's walk, in 16 dimensions.")

---
## Part 5 — Overfitting: zero residual, large error

Fit polynomials of degree 2 and degree 9 (`np.polyfit`) to 10 noisy samples of $y = x^2$.
Compute, for each: the data-residual RMS, and the true-error RMS on a dense grid over $[0, 3]$.

In [ ]:
rng = np.random.default_rng(3)
x_d = np.linspace(0, 3, 10)
y_d = x_d**2 + 0.3*rng.standard_normal(10)
xx  = np.linspace(0, 3, 400)

p2 = ...   # TODO: np.polyfit degree 2
p9 = ...   # TODO: np.polyfit degree 9
res2 = ...; res9 = ...    # TODO: data-residual RMS of each fit
err2 = ...; err9 = ...    # TODO: true-error RMS of each fit vs x^2 on xx
print(f"deg 2: residual {res2:.3f}  error {err2:.3f}")
print(f"deg 9: residual {res9:.3f}  error {err9:.3f}")
plt.plot(xx, xx**2, 'b', label='truth'); plt.plot(x_d, y_d, 'k.')
plt.plot(xx, np.polyval(p2, xx), 'g', label='degree 2')
plt.plot(xx, np.polyval(p9, xx), 'r--', label='degree 9')
plt.ylim(-2, 11); plt.legend(); plt.show()

In [ ]:
# ✅ check — Part 5
assert res9 < res2, "The flexible fit should have the SMALLER data residual..."
assert err9 > err2, "...and the LARGER true error. That is overfitting in two inequalities."
print("OK: residual down, error up. Keep this pair of inequalities in mind forever.")

---
## Closing reflections (markdown answers)

1. In Part 2 residual and error had different *units*; in Part 5 the same units. Why?
2. Where in classical FEM (Lectures 1–2) is there anything like overfitting? (Careful — the
   'data' there is an exact equation. What would have to become noisy for overfitting to return?)
3. Part 4's network fit the *solution* from data. Lecture B's exercise fits it from the
   *equation* — no data at all. Which ingredients of this notebook will survive unchanged?